In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

df = pd.read_csv("earthquake_engineered.csv")
geo_scaler = StandardScaler()
geo_scaled = geo_scaler.fit_transform(df[['Latitude', 'Longitude']])

In [2]:
# K-Means - assumes roughly circular clusters
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(geo_scaled)
km_sil = silhouette_score(geo_scaled, km_labels)
print(f"K-Means (k=6) silhouette score: {km_sil:.3f}")
df['KMeans_Cluster'] = km_labels

K-Means (k=6) silhouette score: 0.575


In [3]:
# DBSCAN - density based, follows the actual shape of fault lines instead of forcing circles
dbscan = DBSCAN(eps=0.2, min_samples=15)
db_labels = dbscan.fit_predict(geo_scaled)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
db_sil = silhouette_score(geo_scaled, db_labels) if n_clusters_db > 1 else float('nan')
print(f"DBSCAN found {n_clusters_db} clusters, {n_noise} noise points ({n_noise/len(df)*100:.1f}%), silhouette = {db_sil:.3f}")
df['DBSCAN_Cluster'] = db_labels

DBSCAN found 4 clusters, 27 noise points (0.1%), silhouette = 0.070


In [4]:
# Visual comparison: K-Means (forced circular boundaries) vs DBSCAN (follows fault-line shape)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(df['Longitude'], df['Latitude'], c=df['KMeans_Cluster'], cmap='tab10', s=5)
axes[0].set_title('K-Means clusters')
axes[1].scatter(df['Longitude'], df['Latitude'], c=df['DBSCAN_Cluster'], cmap='tab10', s=5)
axes[1].set_title('DBSCAN clusters')
plt.savefig('cluster_comparison.png')
plt.close()
print("Cluster comparison plot saved to cluster_comparison.png")

Cluster comparison plot saved to cluster_comparison.png


In [5]:
df.to_csv("earthquake_clustered.csv", index=False)
print("Clustered data saved to earthquake_clustered.csv")

Clustered data saved to earthquake_clustered.csv
